# ZTLF 01 — Ingestion and natural-defect census

This is the step I skipped the first time around: measuring what's actually wrong with each dataset before touching it. Not injected defects — whatever's already there.

Why I'm bothering: my original evaluation injected 850 defects and then recovered exactly 850 of them, because the detection rules were built from the same list used to inject them. Precision and recall of 1.0, but that's a tautology, not a result — any referee reads that as a unit test.

Profiling the natural defects fixes three things at once: it gives me a detection target I didn't hand-pick, it spreads the defect taxonomy across three very different domains (bank / clinical / retail) instead of one, and it means the quality rules come from what's actually documented about each dataset rather than whatever seemed convenient to inject.

Defect taxonomy used throughout:

| Class | Name | Description |
|---|---|---|
| D1 | Structural | parse failures, wrong column count, type-coercion loss |
| D2 | Missingness | nulls and sentinel tokens (`?`, `unknown`, `NA`) |
| D3 | Uniqueness | duplicate rows, duplicate keys |
| D4 | Domain | values outside a documented categorical domain |
| D5 | Range | numeric values outside a plausible range |
| D6 | Consistency | cross-field contradictions |
| D7 | Representation | case / whitespace / abbreviation variants |


In [ ]:
#@title Re-attach to the environment (run notebook 00 first)
import pathlib, sys, os, glob, json

from google.colab import drive
drive.mount('/content/drive')          # each notebook needs its own mount

PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/Paper1")

if not PROJECT_ROOT.exists():
    mydrive = pathlib.Path("/content/drive/MyDrive")
    found = sorted(p.name for p in mydrive.iterdir() if p.is_dir())[:30]
    raise SystemExit(
        f"{PROJECT_ROOT} not found.\n"
        f"Top-level folders in MyDrive: {found}\n"
        "Either run ZTLF_00_environment.ipynb, or edit PROJECT_ROOT above "
        "to match your actual folder name.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

cands = sorted(glob.glob("/usr/lib/jvm/java-11-openjdk*"))
if cands:
    os.environ["JAVA_HOME"] = cands[0]

manifest = PROJECT_ROOT / "outputs/logs/environment_manifest.json"
RANDOM_SEED = json.loads(manifest.read_text())["random_seed"] if manifest.exists() else 20260803
print("project root:", PROJECT_ROOT, "| seed:", RANDOM_SEED)

In [ ]:
#@title Upload the profiling modules to Drive/src (run once, or after edits)
# The two modules ztlf_profiling.py and ztlf_specs.py must live in Paper1/src/.
# If you have not copied them yet, upload them via the Colab file browser or
# use the cell below to verify they are importable.

import importlib
try:
    import ztlf_profiling, ztlf_specs
    importlib.reload(ztlf_profiling); importlib.reload(ztlf_specs)
    print("modules loaded OK")
except ModuleNotFoundError as e:
    raise SystemExit(
        f"{e}\n\nPlace ztlf_profiling.py and ztlf_specs.py in "
        f"{PROJECT_ROOT/'src'} and re-run this cell.")

In [ ]:
#@title Point the specs at your Drive raw-data files
from ztlf_specs import bank_spec, DIABETES_SPEC, online_retail_spec
import dataclasses

RAW = PROJECT_ROOT / "data/raw"

SPECS = [
    bank_spec("bank_marketing_small", str(RAW / "bank.csv")),
    bank_spec("bank_marketing_full",  str(RAW / "bank-full.csv")),
    dataclasses.replace(DIABETES_SPEC, path=str(RAW / "diabetic_data.csv")),
    online_retail_spec(str(RAW / "online_retail_II.csv")),
]

missing = [s.path for s in SPECS if not pathlib.Path(s.path).exists()]
if missing:
    print("MISSING FILES — place these in data/raw/ before continuing:")
    for m in missing:
        print("  ", m)
else:
    print("All", len(SPECS), "source files found.")

### Online Retail II — a format note
UCI ships this one as an `.xlsx` workbook with two sheets (2009–2010 and 2010–2011). If your Drive copy is still the Excel file, run the next cell once to merge it into a single CSV. Already have a CSV? Skip it.


In [ ]:
#@title (Optional) Convert Online Retail II xlsx -> csv
import pandas as pd

xlsx = RAW / "online_retail_II.xlsx"
csv  = RAW / "online_retail_II.csv"

if xlsx.exists() and not csv.exists():
    sheets = pd.read_excel(xlsx, sheet_name=None, dtype=str)
    frames = []
    for sheet_name, sdf in sheets.items():
        sdf["_source_sheet"] = sheet_name
        frames.append(sdf)
    combined = pd.concat(frames, ignore_index=True)
    combined.to_csv(csv, index=False)
    print(f"wrote {csv}  rows={len(combined):,}  sheets={list(sheets)}")
elif csv.exists():
    print("CSV already present:", csv)
else:
    print("No Online Retail II file found in data/raw/")

In [ ]:
#@title Run the natural-defect census across all datasets
from ztlf_profiling import profile_dataset, summarize_census
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

all_results = {}
summaries = []

for spec in SPECS:
    if not pathlib.Path(spec.path).exists():
        print(f"skip {spec.name} (file missing)")
        continue
    print(f"profiling {spec.name} ...")
    res = profile_dataset(spec)
    all_results[spec.name] = res
    summaries.append(summarize_census(res))

census_summary = pd.concat(summaries, ignore_index=True)
census_summary

In [ ]:
#@title Per-dataset detail: missingness
for name, res in all_results.items():
    m = res["missingness"]
    hit = m[m.total_missing > 0]
    print("=" * 90)
    print(name, "| columns with missingness:", len(hit))
    print("=" * 90)
    if len(hit):
        print(hit[["column", "total_missing", "pct_missing",
                   "sentinel_tokens"]].head(12).to_string(index=False))
    else:
        print("none detected")
    print()

In [ ]:
#@title Per-dataset detail: uniqueness, consistency, range, representation
for name, res in all_results.items():
    print("=" * 90); print(name); print("=" * 90)
    for key, cols in [
        ("uniqueness",     ["check", "subject", "n_violations", "pct"]),
        ("consistency",    ["rule", "n_violations", "pct"]),
        ("range",          ["column", "n_violations", "pct",
                            "observed_min", "observed_max"]),
        ("domain",         ["column", "n_violations", "pct", "example_values"]),
        ("representation", ["column", "n_colliding_groups", "example_collisions"]),
    ]:
        d = res[key]
        if d.empty:
            continue
        keep = [c for c in cols if c in d.columns]
        if key == "representation":
            d = d[d.n_colliding_groups > 0]
        elif "n_violations" in d.columns:
            d = d[d.n_violations > 0]
        if len(d):
            print(f"\n-- {key} --")
            print(d[keep].head(12).to_string(index=False))
    print()

In [ ]:
#@title Persist every census table to Drive (these become manuscript tables)
TABLES = PROJECT_ROOT / "outputs/tables"

census_summary.to_csv(TABLES / "T1_natural_defect_census_summary.csv", index=False)

for name, res in all_results.items():
    for key, df in res.items():
        if df.empty:
            continue
        df.to_csv(TABLES / f"census_{name}__{key}.csv", index=False)

# Provenance record: which exact file produced which numbers
prov = pd.concat([r["overview"] for r in all_results.values()], ignore_index=True)
prov.to_csv(TABLES / "T0_dataset_provenance.csv", index=False)

print("wrote census tables to", TABLES)
prov[["dataset", "n_rows", "n_columns", "sha256", "license"]]

### What to look for in the census output
A few questions I keep in mind reading this table, since the answers end up in the manuscript:

- Which defect classes never show up naturally? Those are fair game for injection later — if a class is already common in a dataset, injecting more of it would make the ground truth ambiguous.
- Does one dataset dominate a given defect class? That's basically the evidence against the "you only tested on one dataset" objection.
- Where does a "valid" value double as missingness? Bank Marketing's `unknown` is both a real category and, functionally, a missing-value flag. A naive rule has to pick one interpretation — I think the more honest move is to make that choice explicit rather than let it happen by accident.
- Does deduplicating by key silently destroy real records? In the diabetes data, `patient_nbr` repeats across encounters on purpose. A naive "quarantine duplicate keys" rule would wipe out a large chunk of a clinical dataset.


In [ ]:
#@title Figure 1 — natural defect density by class and dataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

piv = census_summary.pivot(index="defect_class", columns="dataset",
                           values="n_affected_cells").fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(piv.index))
w = 0.8 / max(len(piv.columns), 1)

for i, col in enumerate(piv.columns):
    ax.bar(x + i * w, piv[col].values, w, label=col)

ax.set_xticks(x + w * (len(piv.columns) - 1) / 2)
ax.set_xticklabels(piv.index, rotation=20, ha="right")
ax.set_ylabel("Affected cells / rows (log scale)")
ax.set_yscale("symlog")
ax.set_title("Naturally occurring defects by class and dataset (pre-injection)")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

figpath = PROJECT_ROOT / "outputs/figures/F1_natural_defect_census.png"
fig.savefig(figpath, dpi=300)
print("wrote", figpath)
plt.show()

---
### Next: ZTLF 02
With the natural-defect baseline in hand, notebook 02:

1. Loads each dataset into a Bronze Delta table with provenance metadata.
2. Injects Jenga-style corruption for whichever defect classes don't occur naturally, at contamination rates from 0–30% across multiple seeds — so detection performance is a curve, not a single number I picked once.
3. Records a ground-truth defect mask so precision/recall can be scored for both my own gate and the baseline tools (Great Expectations, Soda Core, PyDeequ, Pandera) later.
